# vast.ai — Notebook 2: TRAIN on the RTX PRO 6000 (RTX-optimised)

Run after Notebook 1. Code auto-scales to the card (auto_bsz->1024 @96GB,
--net-mult 4, --amp, TF32, multi-core BFS).

1. Stage 1 — fill the corpus (breadth; many-core BFS).
2. Train WM big + gated (--net-mult 4 --amp); chg-acc gate stops on plateau.
3. Pretrain the black-box ChangeNet prior big.
4. Representation A/B (wm_augment colour-perm + D4) vs the 0.149 baseline.
5. Commit results back.

In [ ]:
# Cell 1 — RTX setup (PY comes from Notebook 1)
import os, subprocess
PY  = open("/workspace/PY.txt").read().strip() if os.path.exists("/workspace/PY.txt") else __import__("sys").executable
V19 = "/workspace/arc3/CommunitySolutions/chronos_solver/v19"
print(subprocess.run([PY,"-c","import torch;torch.backends.cuda.matmul.allow_tf32=True;torch.backends.cudnn.allow_tf32=True;p=torch.cuda.get_device_properties(0);print('GPU',p.name,round(p.total_memory/1e9),'GB; TF32 on')"],capture_output=True,text=True).stdout)
print("PY =", PY, "| cores:", os.cpu_count())

In [ ]:
# Cell 2 — STAGE 1: fill the corpus (breadth). BFS = CPU/multi-core; runs in the
# background while the GPU trains. Watch CAMPAIGN_LOG.md / games count climb.
import subprocess
subprocess.Popen(f"cd {V19} && while true; do {PY} solve_all.py --bfs-timeout 300 --shuffle; done",
                 shell=True, stdout=open(f"{V19}/solve_all.log","a"), stderr=subprocess.STDOUT)
print("Stage 1 breadth solving started in background")

In [ ]:
# Cell 3 — harvest the corpus into transitions
!cd {V19} && {PY} harvest_wm.py 2>&1 | tail -3

In [ ]:
# Cell 4 — TRAIN the WORLD MODEL big + gated (RTX). The held-out chg-acc gate keeps
# only weights that improve transfer; --patience stops if it plateaus (architecture-bound).
!cd {V19} && {PY} train_wm_v19.py --epochs 80 --net-mult 4 --bsz -1 --amp --patience 8 2>&1 | tail -20

In [ ]:
# Cell 5 — PRETRAIN the black-box ChangeNet prior big (lifts exploration efficiency)
!cd {V19} && {PY} pretrain.py --net-mult 4 --bsz -1 --amp --per_game 1500 --epochs 40 2>&1 | tail -15

In [ ]:
# Cell 6 — EXPERIMENT: representation A/B (colour-perm + D4) vs baseline 0.149
!cd {V19} && {PY} wm_augment.py --epochs 40 --ncolor 4 --d4 2>&1 | grep augA

In [ ]:
# Cell 7 — monitor (re-run any time)
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv,noheader
!echo '--- breadth ---' && cd {V19} && {PY} -c "import json,glob; print('games',len(glob.glob('solutions/*.json')),'levels',sum(len(json.load(open(f))) for f in glob.glob('solutions/*.json')))"
!echo '--- WM_LOG tail ---' && tail -4 {V19}/WM_LOG.md

In [ ]:
# Cell 8 — COMMIT results back (needs GIT_TOKEN in Notebook 1 Cell 1)
!cd /workspace/arc3 && git add CommunitySolutions/chronos_solver/v19/*.py CommunitySolutions/chronos_solver/v19/*.md CommunitySolutions/chronos_solver/v19/*.pt CommunitySolutions/chronos_solver/v19/WM_LOG.md CommunitySolutions/chronos_solver/v19/CAMPAIGN_LOG.md
!cd /workspace/arc3 && git commit -m "rtx: trained WM (net-mult 4, amp) + pretrained prior + breadth corpus" && git push origin main